[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/freeform/freeform.ipynb)

# FreeForm Elasticity

This example simulates an elastic solid, a Stanford bunny, by blending a handful of affine motions. Smooth weight fields spread each motion through the body, so a few dozen controls carry a whole solid through large turns and stretches. The weight fields come from an eigenproblem over the body's volume: they are the gentlest ways a scalar can vary through the material.

Points and displacements are directions, antivectors. The gradient of a weight field is a covector, a vector, which reads off the field's change along a direction by pairing, `gradient & direction`. The local deformation is a map on directions, the identity plus every control displacement paired with its feature's gradient, and its stress maps covectors to covectors. The sampling map's adjugate, its pairing transpose, collects the forces at the volume samples onto the controls.

A short pull on one ear sets the bunny moving: the ear swings out and springs back while the base stays on its compliant support.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, Orientation, stack
from numga.algebras import VGA3D as ga
from examples import MODEL_DIR
from examples.animation import save_figure, show_animation
from examples.mechanics.freeform import geometry, render
from examples.mechanics.freeform.numerics import minimize_state

In [ ]:
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# Points and displacements are antivectors; gradients and forces are covectors, vectors.
Antivector = ga.gatype.antivector()
Vector = ga.gatype.vector()
# A point with an intrinsic one prepended, on which affine functions are linear.
AffinePoint = ga.gatype(ga.subspace.from_grades([0, 2]), orientation=Orientation.intrinsic)
Evaluation = ga.gatype((Scalar, Scalar))
Gradient = ga.gatype((Vector, Scalar))
Sampling = ga.gatype((Antivector, Antivector))
Deformation = ga.gatype((Antivector, Antivector))
Stress = ga.gatype((Vector, Vector))                             # from gradients to forces
Mass = ga.gatype((Vector, Antivector))                           # from accelerations to forces

# The planes across the three axes, and the directions along them, which they pair with to the identity.
planes = stack((mv.x, mv.y, mv.z))                               # [3] Vector
directions = planes.dual()                                       # [3] Antivector
one = mv(Scalar.derive.flipped_orientation, [1.0])              # [] intrinsic Scalar

## 1. A solid inside a triangle surface

The Stanford bunny's closed surface bounds a solid. Samples inside it integrate the elastic energy and the inertia, and a smaller set of centres carries the weight fields; the triangles of the surface show the deformation.

In [ ]:
height = 2
n_quadrature = 1600
n_centers = 96
mesh_path = MODEL_DIR / "bunny.obj"

In [ ]:
vertices, faces, samples, sample_volumes, centers = geometry.load(
    mesh_path, height, n_quadrature, n_centers
)                                                               # Antivector, -, Antivector, -, Antivector
figure = render.draw(vertices, faces)
save_figure(figure, "freeform_rest")
figure

## 2. Smooth weight fields

Reproducing kernels interpolate affine functions exactly, so a uniform stretch stays uniform despite the irregular samples. Each kernel's gradient is a covector, assembled from its change along the three axis directions. The weight fields are the lowest modes of an elastic stiffness against the mass; the constant field is among them and carries every global affine motion.

In Kaolin's PyTorch tensor notation, [the affine features](https://github.com/NVIDIAGameWorks/kaolin/blob/ee8b7eca1e56b1b9fd1509fc406f8d8ccbc9240b/kaolin/physics/simplicits/rkpm.py#L372-L423) are a column of ones concatenated with the coordinates, and their derivative is an identity block below a zero row. The [weight stiffness](https://github.com/NVIDIAGameWorks/kaolin/blob/ee8b7eca1e56b1b9fd1509fc406f8d8ccbc9240b/kaolin/physics/simplicits/rkpm.py#L222-L234) is a weighted transpose product after flattening the sample and vector axes.

In [ ]:
radius = 0.3
n_modes = 9
lame_lambda = 20
lame_mu = 10
density = 1

In [ ]:
def kernels(queries: Antivector, centers: Antivector, radius: float) -> tuple[Evaluation, Gradient]:
    """Maps from scalar nodal values to values and spatial gradients at query points."""
    points = one + centers                                       # [nodes] AffinePoint
    difference = queries[..., None] - centers                    # [queries, nodes] Antivector
    gaussian = (-difference.scalar_norm_squared() / radius**2).exp()  # [queries, nodes] Scalar

    # Correct the Gaussian kernels so their weighted sum reproduces every affine function.
    moment = (gaussian * (points * points.scalar_product(AffinePoint))).sum(axis=-1)  # [queries] AffinePoint <- AffinePoint
    correction = moment.solve(one + queries)                     # [queries] AffinePoint
    reproduction = points.scalar_product(correction[..., None])  # [queries, nodes] Scalar
    weights = gaussian * reproduction                           # [queries, nodes] Scalar

    # Differentiating the moment equation accounts for the correction changing with position.
    gaussian_derivative = gaussian * (-2 / radius**2) * difference.dot(Antivector)  # [queries, nodes] Scalar <- Antivector
    moment_change = (points * gaussian_derivative * reproduction).sum(axis=-1)  # [queries] AffinePoint <- Antivector
    correction_derivative = moment.solve(Antivector - moment_change)  # [queries] AffinePoint <- Antivector
    derivatives = gaussian_derivative * reproduction + gaussian * points.scalar_product(correction_derivative[..., None])  # [queries, nodes] Scalar <- Antivector
    # Each weight's gradient: the covector that reads its change along every direction.
    gradients = (planes * derivatives[..., None](directions)).sum(axis=-1)   # [queries, nodes] Vector

    # The node axis becomes the input field of the interpolation and gradient maps.
    return (weights * Scalar).field(1), (gradients * Scalar).field(1)

evaluation, gradient = kernels(samples, centers, radius)
# Integrating the paired responses gives two maps on scalar nodal fields.
values_at_samples = evaluation.field(0)                          # Scalar[samples] <- Scalar[nodes]
gradients_at_samples = gradient.field(0)                         # Vector[samples] <- Scalar[nodes]
weight_mass = values_at_samples.adjoint()((evaluation * sample_volumes).field(0))  # Scalar[nodes] <- Scalar[nodes]
# Summing the rest elastic costs of three translation directions gives this weight.
weight_stiffness = gradients_at_samples.adjoint()(
    (gradient * sample_volumes * (lame_lambda + 4 * lame_mu)).field(0)
)                                                               # Scalar[nodes] <- Scalar[nodes]
eigenvalues, modes = weight_stiffness.eigh(weight_mass)           # [nodes] Scalar, Scalar[nodes]
modes = modes[:n_modes]                                          # [modes] Scalar[nodes]

## 3. Blending affine motions

Four control points, the corners of a tetrahedron around the bunny, determine an affine motion through their barycentric coordinates, and each weight field carries its own four control displacements. The same sampling map, from control displacements to displacements at points, moves the volume samples and the displayed surface. The mass pairs each sample's displacement with its metric covector, `sampling.dual()`, and collects the result onto the controls with the sampling map's adjugate.

In the matrix notation of the FreeForm evaluation code, [reduced inertia](https://github.com/nv-tlabs/freeform-eval/blob/a969a62f765c5bed5b23258321afddaf3f171bd8/sim/simplicits_ext.py#L361-L370) is $B^T M B$, with each sample mass repeated across its spatial coordinates, and [sampled forces return to the controls](https://github.com/nv-tlabs/freeform-eval/blob/a969a62f765c5bed5b23258321afddaf3f171bd8/sim/simplicits_ext.py#L590-L599) through $B^T$.

In [ ]:
corners = mv.vector([[-1, -1, 0], [1, -1, 2], [-1, 1, 2], [1, 1, 0]]).dual()   # [4] Antivector
n_controls = len(corners) * n_modes

In [ ]:
def deformation_basis(queries: Antivector, centers: Antivector, radius: float,
                      modes: Scalar, corners: Antivector) -> tuple[Sampling, Vector]:
    """The displacement map and feature gradients for four affine controls per weight mode."""
    evaluation, gradient = kernels(queries, centers, radius)
    weights = evaluation[:, None](modes)                         # [samples, modes] Scalar
    weight_gradients = gradient[:, None](modes)                   # [samples, modes] Vector

    # Four control points determine each affine map through their barycentric coordinates.
    points = one + corners                                      # [4] AffinePoint
    moment = (points * points.scalar_product(AffinePoint)).sum(axis=0)  # AffinePoint <- AffinePoint
    coordinates = points.scalar_product(moment.solve(AffinePoint))    # [4] Scalar <- AffinePoint
    affine_weights = coordinates(one + queries[:, None])         # [samples, 4] Scalar
    # Each coordinate's gradient: the covector that reads its change along every direction.
    affine_gradients = (planes[:, None] * coordinates[None, :](directions[:, None])).sum(axis=0)   # [4] Vector

    features = weights[..., None] * affine_weights[:, None, :]   # [samples, modes, 4] Scalar
    derivatives = (weight_gradients[..., None] * affine_weights[:, None, :]
                   + weights[..., None] * affine_gradients)     # [samples, modes, 4] Vector
    # A state is the field of the four control displacements of every affine map.
    sampling = (features.reshape(len(queries), -1) * Antivector).field(1)  # [samples] Antivector <- Antivector[controls]
    gradients = derivatives.reshape(len(queries), -1).field()    # [samples] Vector[controls]
    return sampling, gradients

sampling, gradients = deformation_basis(samples, centers, radius, modes, corners)
surface, _ = deformation_basis(vertices, centers, radius, modes, corners)
# Each sample's displacement as the covector the metric pairs with it, collected onto the controls.
mass = sampling.field(0).adjugate()((sampling.dual() * sample_volumes * density).field(0))  # Vector[controls] <- Antivector[controls]

## 4. Elastic energy and stress

The local deformation measures how neighbouring points separate. Stretching and changing volume cost energy; a rigid turn costs none. The stress is the energy's derivative, a map from a feature's gradient to the force it carries. Its stretch part goes through the metric, `local.adjoint().adjugate()`, and its volume part is the cofactor, which carries the plane spanned by two directions, their join `a & b`, to the plane spanned by their images. Built from joins, the cofactor stays defined when the deformation flattens the solid and its determinant vanishes.

In the matrix notation of Kaolin's Warp code, [the cofactor in the stress](https://github.com/NVIDIAGameWorks/kaolin/blob/ee8b7eca1e56b1b9fd1509fc406f8d8ccbc9240b/kaolin/physics/materials/neohookean_elastic_material.py#L77-L87) is $\det(F)F^{-T}$.

In [ ]:
# Each plane is spanned by the two directions after its own.
cyclic = np.array([[0, 1, 2], [1, 2, 0], [2, 0, 1]])
# The reference volume, the join of the three directions: an intrinsic scalar of value one.
unit_volume = directions[0] & directions[1] & directions[2]


def deformation(state: Antivector, gradients: Vector) -> Deformation:
    """The deformation of the blended affine motions: the identity plus each control displacement
    paired with its feature's gradient."""
    return Antivector + (state * (gradients & Antivector)).sites.sum()

def cofactor(local: Deformation) -> Stress:
    """The map on covectors that carries each plane, spanned by two directions, to the plane spanned by
    their images, over the reference volume."""
    images = local[..., None](directions)                        # [..., 3] Antivector
    spans = images[..., cyclic[:, 1]] & images[..., cyclic[:, 2]]  # [..., 3] intrinsic Vector
    return (spans * (Vector & directions[cyclic[:, 0]])).sum(axis=-1) / unit_volume

def constitutive(local: Deformation, lame_lambda: float, lame_mu: float) -> tuple[Scalar, Stress]:
    """Stable Neo-Hookean energy density and its stress, from gradients to forces."""
    determinant = local.det()                                   # [samples] Scalar
    bulk = lame_lambda + lame_mu
    target = 1 + lame_mu / bulk
    rest_energy = bulk * (1 - target)**2 + 3 * lame_mu
    stretch = local.adjoint()(local).trace()                     # [samples] Scalar
    energy = (bulk * (determinant - target).squared() + lame_mu * stretch - rest_energy) / 2
    # The stretch is metric, so its stress carries the deformation over to covectors through the metric.
    stress = lame_mu * local.adjoint().adjugate() + bulk * (determinant - target) * cofactor(local)   # [samples] Stress
    return energy, stress

def potential(state: Antivector, sampling: Sampling, gradients: Vector,
              sample_volumes: np.ndarray, lame_lambda: float, lame_mu: float,
              clamp: Scalar, load: Vector) -> tuple[Scalar, Vector]:
    """Elastic energy, a compliant base clamp and an applied force density, with their gradient, the
    forces on the controls."""
    displacement = sampling(state)                              # [samples] Antivector
    local = deformation(state, gradients)                        # [samples] Antivector <- Antivector
    density, stress = constitutive(local, lame_lambda, lame_mu)
    density = density + clamp * displacement.dot(displacement) / 2 - (load & displacement)
    energy = (density * sample_volumes).sum(axis=0)               # Scalar
    elastic = (stress(gradients) * sample_volumes).sum(axis=0)     # Vector[controls]
    # The clamp pulls back along the metric covector of the displacement; the adjugate of the sampling
    # collects every local force onto the affine controls.
    applied = sampling.field(0).adjugate()(
        ((clamp * displacement.dual() - load) * sample_volumes).field()
    )                                                           # Vector[controls]
    return energy, elastic + applied

## 5. Pull and release

A short force pulse pulls one ear sideways while a compliant support holds the base. After the release, the stored elastic energy drives the recoil, and implicit Euler damps the motion as it settles. Each step minimizes the elastic and applied energy plus the inertia of departing from free flight. This cell takes about half a minute, simulating and drawing.

In [ ]:
clamp_depth = 0.12
clamp_stiffness = 500
pull_center = mv.vector([-0.25, 0.34, 1.92]).dual()             # Antivector
pull_width = 0.23
pull_force = (mv.x + 0.7 * mv.y) * 0.6                           # Vector
pull_duration = 0.4
dt = 0.05
n_steps = 80
n_iterations = 30
tolerance = 1e-7
duration_ms = round(dt * 1000)

In [ ]:
clamp = (-(mv.z & samples).squared() / clamp_depth**2).exp() * clamp_stiffness   # [samples] Scalar
profile = (-(samples - pull_center).scalar_norm_squared() / pull_width**2).exp()  # [samples] Scalar
# Normalized over the solid, so that the profile distributes the stated total force.
load = profile * pull_force / (profile * sample_volumes).sum(axis=0)  # [samples] Vector
loading = np.sin(np.pi * np.minimum(np.arange(n_steps) * dt / pull_duration, 1))**2


def trajectory() -> Iterator[Antivector]:
    state = mv.vector([[0, 0, 0]] * n_controls).dual().field()   # Antivector[controls]
    velocity = state                                            # Antivector[controls]
    yield vertices + surface(state)
    for strength in loading:
        predicted = state + velocity * dt                       # Antivector[controls]

        def objective(trial: Antivector) -> tuple[Scalar, Vector]:
            energy, force = potential(trial, sampling, gradients, sample_volumes,
                                      lame_lambda, lame_mu, clamp, load * strength)
            # Inertia resists departing from the free-flight prediction.
            change = trial - predicted                          # Antivector[controls]
            inertial_force = mass(change) / dt**2                 # Vector[controls]
            return energy + (inertial_force & change).sites.sum() / 2, force + inertial_force

        following = minimize_state(state, objective, mass / dt**2, n_iterations, tolerance)  # Antivector[controls]
        velocity = (following - state) / dt                      # Antivector[controls]
        state = following                                       # Antivector[controls]
        yield vertices + surface(state)


surfaces = stack(list(trajectory()))                             # [steps + 1, vertices] Antivector
show_animation(render.frames(surfaces, faces), "freeform_bunny", duration_ms)

In [ ]:
# checks
# The ear follows the pull and springs back after release, while the base moves a tenth as far.
displacement = surfaces - surfaces[0]                            # [steps + 1, vertices] Antivector
pulled = (pull_force & displacement).to_array()                  # [steps + 1, vertices]
distances = displacement.scalar_norm_squared().to_array() ** 0.5  # [steps + 1, vertices]
base = (mv.z & surfaces[0]).to_array() < 0.05                    # [vertices]
assert pulled.max() > 0.3 and pulled[-1].max() < 0.05 * pulled.max()
assert distances[:, base].max() < 0.1 * distances.max()

The weight construction and reduced elastic model follow Xiang, Modi and colleagues' [*FreeForm: Reduced-Order Deformable Simulation from Particle-Based Skinning Eigenmodes*](https://research.nvidia.com/labs/sil/projects/freeform/). The time step minimizes their incremental energy with L-BFGS; the paper uses Newton steps. The [Stanford bunny](https://graphics.stanford.edu/data/3Dscanrep/) supplies the closed triangle surface.